In [1]:
%reload_ext autoreload
%autoreload 2

import warnings, os, sys, re
sys.setrecursionlimit(1000000)
warnings.filterwarnings('ignore')
from IPython.display import clear_output

import numpy as np
import pandas as pd
from functools import reduce
from tqdm.auto import tqdm

from rdkit import Chem
from rdkit.Chem import PandasTools
PandasTools.RenderImagesInAllDataFrames(images=True)
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
from rdkit.DataStructs import TanimotoSimilarity
from rdkit.Chem.AllChem import GetMorganFingerprintAsBitVect
from rdkit.Chem import Descriptors

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 600, 400
IPythonConsole.drawOptions.addAtomIndices = False

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook"
pio.renderers.default_config = {"toImageButtonOptions": {"format": "png", "scale": 5}}

sys.path.append(os.path.abspath("C:\\Users\\Idener\\Sync\\CODE\\AIDDTools"))
from fingerprint_tools import calculate_fp, get_similarity

# sys.path.append(os.path.abspath(".."))
# from new_geass import Geass
clear_output()

In [5]:
df_cpd = pd.read_csv('../data/Database_KEGG_cpd.csv')
df_cpd = df_cpd.iloc[20:].dropna(subset=['FORMULA', 'INCHI', 'REACTION'])
df_cpd = df_cpd[df_cpd['MOL_WEIGHT'] > 50]

allowed_elements = R'(C|H|N|O|F|Cl|Br|P|S)'
pattern = rf'^({allowed_elements}[0-9]*)+$'
df_cpd = df_cpd[
    df_cpd['FORMULA'].str.fullmatch(pattern, case=False, na=False) &
    df_cpd['FORMULA'].str.contains('C', case=False, na=False) &
    df_cpd['FORMULA'].str.contains('H', case=False, na=False)]

df_cpd

KeyError: ['FORMULA', 'INCHI', 'REACTION']

In [ ]:
df_rn = pd.read_parquet(R"C:\Users\Idener\Sync\CODE\fantastic-biofragmenter\data\Database_KEGG_rn.gzip", engine='fastparquet')
df_rn = df_rn.explode('RPAIR')
df_rn['RPAIR'] = df_rn['RPAIR'].apply(lambda x: x.split('_'))
df_rn['RPAIR'] = df_rn['RPAIR'].apply(lambda x: [df_cpd.loc[df_cpd['ENTRY'] == r, 'ISOSMILES'].values[0]
                                                 if not df_cpd.loc[df_cpd['ENTRY'] == r, 'ISOSMILES'].empty
                                                 else None for r in x])
df_rn = df_rn[df_rn['RPAIR'].apply(lambda x: all(x))]

df_rn[['RPAIR_1', 'RPAIR_2']] = pd.DataFrame(df_rn['RPAIR'].tolist(), index=df_rn.index)
PandasTools.AddMoleculeColumnToFrame(df_rn, smilesCol='RPAIR_1', molCol='ROMOL_1')
PandasTools.AddMoleculeColumnToFrame(df_rn, smilesCol='RPAIR_2', molCol='ROMOL_2')
df_rn = df_rn.dropna(subset=['ROMOL_1', 'ROMOL_2'])

df_rn['RPAIR_1_ATOMS'] = df_rn['ROMOL_1'].apply(lambda x: x.GetNumHeavyAtoms() if x else None)
df_rn['RPAIR_2_ATOMS'] = df_rn['ROMOL_2'].apply(lambda x: x.GetNumHeavyAtoms() if x else None)
df_rn['ATOM_DIFF'] = abs(df_rn['RPAIR_1_ATOMS'] - df_rn['RPAIR_2_ATOMS'])
df_rn = df_rn[df_rn['ATOM_DIFF'] <= 5]

df_rn = df_rn[['REACTION', 'EQUATION', 'BRITE', 'RPAIR_1', 'RPAIR_2', 'ATOM_DIFF', 'ROMOL_1', 'ROMOL_2']]

In [ ]:
methods = ['maccs', 'rdkit', 'morgan2', 'morgan3', 'avalon', 'torsion', 'mhfp', 'mhfp6']

for method in methods:
    df_rn['FP_1'] = df_rn['ROMOL_1'].apply(lambda mol: calculate_fp(mol, method=method))
    df_rn['FP_2'] = df_rn['ROMOL_2'].apply(lambda mol: calculate_fp(mol, method=method))
    df_rn[f'similarity_{method}'] = df_rn.apply(lambda row: get_similarity(row['FP_1'], row['FP_2'], method=method), axis=1)

df_rn.drop(columns=['FP_1', 'FP_2', 'ROMOL_1', 'ROMOL_2']).to_csv(R"C:\Users\Idener\Desktop\KEGG_RPAIRS.csv", index=False)
df_rn

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

bin_edges = np.arange(0, 1.1, 0.1)
bin_labels = [f"{round(bin_edges[i], 1)}–{round(bin_edges[i+1], 1)}" for i in range(len(bin_edges)-1)][::-1]

# Determine number of rows and columns for subplots (2x3 grid here)
rows = 2
cols = 4

# Calculate counts for all methods first to find the global max
all_counts = []
for method in methods:
    df_1 = df_rn.copy().dropna(subset=['BRITE'])
    df_1 = df_1[df_1['BRITE'].str.contains('5. Isomerase reactions')]
    similarities = df_1[f'similarity_{method}'].dropna()
    counts, _ = np.histogram(similarities, bins=bin_edges)
    counts = counts[::-1]  # reverse to match reversed x-axis
    all_counts.append(counts)
max_count = max([max(counts) for counts in all_counts])

fig = make_subplots(rows=rows, cols=cols, subplot_titles=[m.upper() for m in methods])

for i, method in enumerate(methods):
    row = i // cols + 1
    col = i % cols + 1

    counts = all_counts[i]

    fig.add_trace(
        go.Bar(x=bin_labels, y=counts, name=method.upper(), showlegend=False),
        row=row, col=col
    )

    fig.update_xaxes(title_text="Similarity", categoryorder='array', categoryarray=bin_labels, row=row, col=col)
    fig.update_yaxes(title_text="Count", range=[0, max_count], row=row, col=col)  # same y-axis max

fig.update_layout(
    title="Similarity Distributions per Fingerprint Method",
    height=600,
    width=1000,
    template="plotly_white"
)

fig.show()

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Define thresholds
thresholds = np.arange(0.0, 1.01, 0.05)

# Collect correlation data
results = {method: {'thresholds': [], 'pearson': [], 'spearman': []} for method in methods}

for method in methods:
    for threshold in thresholds:
        subdf = df_rn[df_rn[f'similarity_{method}'] >= threshold].dropna(subset=['ATOM_DIFF'])
        if subdf.shape[0] >= 5:
            corr_pearson = subdf[[f'similarity_{method}', 'ATOM_DIFF']].corr(method='pearson').iloc[0, 1]
            corr_spearman = subdf[[f'similarity_{method}', 'ATOM_DIFF']].corr(method='spearman').iloc[0, 1]
        else:
            corr_pearson = np.nan
            corr_spearman = np.nan

        results[method]['thresholds'].append(threshold)
        results[method]['pearson'].append(abs(corr_pearson))
        results[method]['spearman'].append(abs(corr_spearman))

# --- Pearson Plot ---
fig_pearson = go.Figure()
for method in methods:
    fig_pearson.add_trace(go.Scatter(
        x=results[method]['thresholds'],
        y=results[method]['pearson'],
        mode='lines+markers',
        name=method.upper()
    ))

fig_pearson.update_layout(
    title='|Pearson Correlation| vs Similarity Threshold',
    xaxis_title='Similarity Threshold (Descending)',
    yaxis_title='|Pearson r|',
    xaxis=dict(autorange='reversed'),
    template='plotly_white',
    width=900,
    height=500
)

# --- Spearman Plot ---
fig_spearman = go.Figure()
for method in methods:
    fig_spearman.add_trace(go.Scatter(
        x=results[method]['thresholds'],
        y=results[method]['spearman'],
        mode='lines+markers',
        name=method.upper()
    ))

fig_spearman.update_layout(
    title='|Spearman Correlation| vs Similarity Threshold',
    xaxis_title='Similarity Threshold (Descending)',
    yaxis_title='|Spearman ρ|',
    xaxis=dict(autorange='reversed'),
    template='plotly_white',
    width=900,
    height=500
)

# Show both plots
fig_pearson.show()
fig_spearman.show()


In [ ]:
from skfp.fingerprints import MACCSFingerprint, PubChemFingerprint

smiles_list = ["O=S(=O)(O)CCS(=O)(=O)O", "O=C(O)c1ccccc1O"]
mols = [Chem.MolFromSmiles(smiles) for smiles in smiles_list]


# fp_maccs = MACCSFingerprint(n_jobs=-1)
# fp_maccs_count = MACCSFingerprint(count=True, n_jobs=-1)

fp_pubchem = PubChemFingerprint(n_jobs=-1)
fp_pubchem_count = PubChemFingerprint(count=True, n_jobs=-1)

# X_maccs = fp_maccs.transform(mols)
# X_maccs_count = fp_maccs_count.transform(mols)

X_pubchem = fp_pubchem.transform(mols)
# X_pubchem_count = fp_pubchem_count.transform(mols)
X_pubchem

In [ ]:
sys.path.append(os.path.abspath("C:\\Users\\Idener\\Documents\\MEGA\\drug_design_modules\\NC-MFP"))

from itertools import chain
from NC_MFP_Algorithm.Preprocessing_step import Preprocessing
from NC_MFP_Algorithm.Scaffold_matching_step import Scaffold_Matching
from NC_MFP_Algorithm.Fragment_list_generation_step import Fragment_List_Generation
from NC_MFP_Algorithm.SFCP_assigning_step import SFCP_Assigning
from NC_MFP_Algorithm.Fragment_identifying_step import Fragment_identifying
from NC_MFP_Algorithm.Fingerprint_representation_step import Fingerprint_representation

# Define classes
Step_1 = Preprocessing()
Step_2 = Scaffold_Matching()
Step_3 = Fragment_List_Generation()
Step_4 = SFCP_Assigning()
Step_5 = Fragment_identifying()
Step_6 = Fingerprint_representation()

All_Scaffold_FilePath = ('C:\\Users\\Idener\\Documents\\MEGA\\drug_design_modules\\NC-MFP\\Data\\All_Optimized_Scaffold_List.txt')

# qMols_Smarts = Step_1.get_Query_Mols_Smarts(Query_FilePath)
qMols_Smarts = df_rn['ROMOL_1'].apply(Chem.MolToSmarts).dropna().tolist()[:500]

Final_all_Fragment_List = []
for qMol_Smart in qMols_Smarts:
    fragments = Step_3.generate_all_Fragment_List(All_Scaffold_FilePath, qMol_Smart) or [qMol_Smart]
    Final_all_Fragment_List.append(fragments)
Final_all_Fragment_List = list(set(chain.from_iterable(Final_all_Fragment_List)))
Final_all_Fragment_Dic = Step_3.generate_all_Fragment_Dictionary(Final_all_Fragment_List)

Final_all_NC_MFP_Info = []
for smarts in tqdm(qMols_Smarts):
    # Step 2: Scaffold matching
    scaffold_matches = Step_2.match_All_Scaffold_Smarts(All_Scaffold_FilePath, smarts)
    scaffold_matches = list(set(scaffold_matches))  # Deduplicate

    # Step 4: Scaffold-Fragment Connection Pairs (SFCP)
    sfcp_list = Step_4.assign_all_sfcp_smarts(All_Scaffold_FilePath, smarts)

    # Step 5: Fragment identification
    try:
        fragment_list = Step_5.identify_Fragment_Smarts(All_Scaffold_FilePath, smarts, Final_all_Fragment_Dic)
    except: pass

    # Merge NC-MFP info
    merged_info = scaffold_matches + sfcp_list + fragment_list
    Final_all_NC_MFP_Info.append(merged_info)


# [6] Fingerprint representation step #
# Get all NC-MFP label
Final_all_NC_MFP_Label = Step_6.get_all_NC_MFP_Label(Final_all_NC_MFP_Info)

# Get all NC-MFP Bit String
Final_Bitstring = Step_6.get_qMol_NC_MFP_Value_Idx(Final_all_NC_MFP_Info, Final_all_NC_MFP_Label)
print(Final_Bitstring)